Imports

In [16]:
import streamlit as st
import re
import requests
import geocoder
import time
from openai import OpenAI

connectie LM studio

In [17]:
client = OpenAI(
base_url = "http://127.0.0.1:1234/v1",
api_key="lm-studio"
)

model inladen

In [18]:
model_name = "nvidia/nemotron-3-nano-4b"

lijst maken van ingredienten op voorraad

In [19]:
items = []

print('')

while True:
    item = input("welke gerechten heb je al/ heb je die je wilt gebruiken. (type stop om te stoppen): ")

    if item == "stop":
        break

    items.append(item)

locatie opvragen

In [20]:
g = geocoder.ip("me")
regio = g.city
lat = g.latlng[0]
lon = g.latlng[1]

Promptstructuur

In [21]:
prompts = [
    f"""
Geef exact 5 gerechten op basis van deze ingrediënten: {items}.

vaste opzet:
gerecht nr|gerecht|ingredienten|duur|nieuwe ingredienten|

Voor elk gerecht:
- bereidingstijd <= 30 minuten
- extra ingrediënt
- en zet letterlijk dit aan het einde van de text zodat ik hem er later uit kan halen: 'nieuw ingredient = (ingredient)'
"""
]

Startijd

In [22]:
for prompt in prompts:
    start_time = time.time()

Data opslaan

In [23]:
data = {
    "model": model_name,
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}


antwoord opvragen

In [29]:
response = client.chat.completions.create(
        model =model_name,
        messages = [
        {"role": "user", "content": prompt}
    ],
    temperature= 0.7
    )

end_time = time.time()
result = response.json()

answer = response.choices[0].message.content
response_length = len(answer)

print(answer)

print("\nMetadata:")
print(f"Modelnaam: {model_name}")
print(f"Lengte response: {response_length} karakters")
print(f"Tijd: {end_time - start_time:.2f} seconden")
print("-" * 50)



1|Kipje risthandvlees|kip, rijst, paprika|25 min|nieuw ingredient = (paprika)  
2|Rijst met Kip en Zucchini|kip, rijst, zucchini|30 min|nieuw ingredient = (zucchini)  
3|Sauspot Kikkerbroodje|kip, rijst, ui|28 min|nieuw ingredient = (ui)  
4|Kipgerecht Rijst met Appel|kip, rijst, appelsap|30 min|nieuw ingredient = (appelsap)  
5|Rijst met Kip en Griekse Kaas|kip, rijst, griekse kaas|27 min|nieuw ingredient = (griekse kaas)

Metadata:
Modelnaam: nvidia/nemotron-3-nano-4b
Lengte response: 427 karakters
Tijd: 500.00 seconden
--------------------------------------------------


C:\Users\coolj\AppData\Local\Temp\ipykernel_39840\2561775929.py:10: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  result = response.json()


???

In [24]:
for answe in answer.splitlines():
    match = re.search(r"nieuw ingrediënt\s*=\s*\((.*?)\)", answe)

    if match:
        ingredient = match.group(1)
        url = f"https://google.com/search?q=supermarkt+voor+{ingredient}+-ai"
        answe += f"|{url}"

    print(answe)



1|Aardappel en karnofstokensoep met appelmoes|aardappel, karbonade, appelmoes|25 min|nieuw ingredient = (zucchini)  
2|Karnof en aardappel stoffrit met appelmoes en komkommer|aardappel, karbonade, appelmoes|30 min|nieuw ingredient = (komkommer)  
3|Appelmoes kipgratin met aardappel en paprika|aardappel, appelmoes, paprika|28 min|nieuw ingredient = (paprika)  
4|Aardappel, karnof en appelmoes salade met basilicum|aardappel, karbonade, appelmoes|25 min|nieuw ingredient = (basilicum)  
5|Karnofstokensoep met appelmoes en citroen|aardappel, karbonade, appelmoes|30 min|nieuw ingredient = (citeroen)
